In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width',200)
pd.set_option("display.max_colwidth", 400)
plt.rcParams['font.family'] = 'Liberation Sans'

# Define SFPQ target

In [ ]:
targ = pd.read_csv("./Brain_SFPQ_target_genes_250603.csv")

bed = pd.read_table("./gencode.vM35.annotation_ensemblcan.bed",names=["chr","start","end","ID","score","strand","start2","end2","exons","blocks","bl_size","bl_start"])
bed["Symbol"] = bed["ID"].str.split(":",expand=True)[0]
bed["tx_length"] = bed["end"]-bed["start"]

def calc_max_intron(row):
    block_count = int(row['blocks'])
    # 単一エキソンの場合はイントロンは存在しないので 0 を返す
    if block_count <= 1:
        return 0
    # blockSizes, blockStarts はカンマ区切りの文字列なのでリストへ変換
    # 末尾に余分なカンマがある場合を考慮して rstrip() で除去
    sizes = list(map(int, row['bl_size'].rstrip(',').split(',')))
    starts = list(map(int, row['bl_start'].rstrip(',').split(',')))
    # 各イントロン長を計算：次のエキソン開始位置 - （現在のエキソン開始位置 + 現在のエキソンサイズ）
    intron_lengths = [starts[i+1] - (starts[i] + sizes[i]) for i in range(block_count - 1)]
    return max(intron_lengths)

# 各行に対して最大イントロン長を計算し、新しい列 "max_intron_length" に追加
bed['max_intron_length'] = bed.apply(calc_max_intron, axis=1)

print("# of input genes",len(targ))
display(pd.DataFrame(targ["Type"].value_counts()))
print("Filtered genes")
filt = set(targ.query('Filt=="y"')["Symbol"])
for gene in sorted(filt):
    print(gene)
targ = targ.query('Filt=="n"')
print("# of genes after filtering protocadherin genes",len(targ))
targ2 = pd.merge(bed[["chr","start","end","strand","Symbol","tx_length","max_intron_length"]],targ,on="Symbol",how="right")
print("# of unmatched symbols:",targ2["chr"].isnull().sum())
print(",".join(targ2[targ2["chr"].isnull()]["Symbol"].to_list()))
targ2.to_csv("./Brain_SFPQ_target_genes_250603_ensemcan_tx.csv")

targ2 = targ2[~targ2["start"].isnull()]
print("# of matched genes",len(targ2))
targ2 = targ2.query('tx_length>=100000')
print("# of long genes",len(targ2))
targ2 = targ2.query('Type=="Strong_Target" & max_intron_length>=45000 | Type=="Weak_Target" & max_intron_length>=30000 | Type=="Non_Target_Long_Genes" & max_intron_length>=30000 | Type=="Long_Genes_without_Long_Intron" & max_intron_length<30000')
print("# of genes with correct intron length classification",len(targ2))
display(pd.DataFrame(targ2["Type"].value_counts()))

targ3 = targ2.copy()
targ3["score"] = 0
targ3[["start","end"]] = targ3[["start","end"]].astype(int)
targ3[["chr","start","end","Symbol","score","strand","Type"]].to_csv("./Brain_SFPQ_target_genes_250603_processed_mm39.bed",header=False,index=False,sep="\t")
! bedtools intersect -a ./Brain_SFPQ_target_genes_250603_processed_mm39.bed -b ./mm39.excluderanges.bed -v > ./Brain_SFPQ_target_genes_250603_processed_mm39_ext.bed

targ3 = pd.read_table("./Brain_SFPQ_target_genes_250603_processed_mm39_ext.bed",names=["chr","start","end","Symbol","score","strand","Type"])
print("# of genes outside of blacklist",len(targ3))
display(pd.DataFrame(targ3["Type"].value_counts()))
print("minimum gene length(bp):",(targ3["end"]-targ3["start"]).min())

internal=2500

targ3[["start","end"]] = targ3[["start","end"]].astype(int)
targ3["internal_start"] = (targ3["start"]+internal).mask(targ3["strand"]=="-",targ3["start"])
targ3["internal_end"] = (targ3["end"]-internal).mask(targ3["strand"]=="+",targ3["end"])

leng_bool = (targ3["internal_end"] - targ3["internal_start"])>=1000
print("# of genes in internal TSS list",len(targ3[leng_bool]))
display(pd.DataFrame(targ3[leng_bool]["Type"].value_counts()))

out_dir = "./"
types = ["Strong_Target","Non_Target_Long_Genes","Long_Genes_without_Long_Intron"]
outputs = ["SFPQ_brain_strong_target_mm39_250603.bed","SFPQ_brain_nontarg_long_mm39_250603.bed","SFPQ_brain_long_wo_long_int_mm39_250603.bed"]

for t,o in zip(types,outputs):
    out = out_dir+o.split(".bed")[0]+"_internal_2500bp.bed"
    targ3[leng_bool].query('Type==@t')[["chr","internal_start","internal_end","Symbol","score","strand"]].to_csv(out,header=False,index=False,sep="\t")
    print(out,"was created")

targ3[["chr","start","end","Symbol","score","strand","Type"]].to_csv(out_dir+"SFPQ_brain_target_mm39_250603.txt",index=False,sep="\t")
targ3[leng_bool][["chr","internal_start","internal_end","Symbol","score","strand","Type"]].to_csv(out_dir+"SFPQ_brain_target_internal_mm39_250603.txt",index=False,sep="\t")

print(out_dir+"SFPQ_brain_target.txt was created")

## >>>These Output Files Were Renamed for "local_metagene_int.sh"
### "../SFPQ/BELD/SFPQ_brain_strong_target_mm39_250603_internal_2500bp.bed" -> Strong_target_int
### "../SFPQ/BELD/SFPQ_brain_nontarg_long_mm39_250603_internal_2500bp.bed" -> Non_target_int
### "../SFPQ/BELD/SFPQ_brain_long_wo_long_int_mm39_250603_internal_2500bp.bed" -> Without_long_intron_int

# Make TSS internal GTF file for FeatureCounts

In [4]:
import pandas as pd
import csv

# GTF のカラム定義
columns = ["chrom", "source", "feature", "start", "end", "score", "strand", "frame", "attributes"]

# GTF ファイルを読み込み（コメント行をスキップ）
gtf_file = "./gencode.vM35.annotation_ensemblcan.gtf"
output_gtf = "./gencode.vM35.annotation_ensemblcan_internal.gtf"

df = pd.read_csv(gtf_file, sep="\t", comment="#", header=None, names=columns)
df = df.query('feature=="transcript"').copy()
print("# of genes:",len(df))
# TSS を -2.5kb 内側にシフト
def adjust_tss(row):
    if row["strand"] == "+":  # プラスストランドの場合
        row["start"] = row["start"] + 2500
    else:  # マイナスストランドの場合
        row["end"] = row["end"] - 2500
    return row

df = df.apply(adjust_tss, axis=1)

# transcript の長さを計算し、1kb 未満のものを除外
df["length"] = df["end"] - df["start"]
df_filtered = df[df["length"] >= 1000].copy()

# 保存
df_filtered.drop(columns=["length"], inplace=True)
df_filtered.to_csv(output_gtf, sep="\t", index=False, header=False, quoting=csv.QUOTE_NONE)
print(output_gtf+" was created")
print("# of output genes:",len(df_filtered))

# of genes: 57132
../Ref/gencode.vM35.annotation_ensemblcan_internal.gtf was created
# of output genes: 26969


# Calc RPKM

In [5]:
bed = pd.read_table("./gencode.vM35.annotation_ensemblcan.bed",names=["chr","start","end","ID","score","strand","start2","end2","exons","blocks","bl_size","bl_start"])
bed[["Geneid","Symbol"]] = bed["ID"].str.split(":",expand=True)[[1,0]]
display(bed.head(2))

def calc_rpkm(inp_path,samples,mapped_reads):
    df = pd.read_table(inp_path,comment="#")
    df = bed[["Symbol","Geneid"]].merge(df,on="Geneid")
     
    for s,m in zip(samples,mapped_reads):
        df["RPM_"+s.split(".bam")[0]] = df[s]*(1000000/m)
        df["RPKM_"+s.split(".bam")[0]] = df[s]*(1000000/m)*(1000/df["Length"])
    df.to_csv(inp_path.split(".txt")[0]+"_RPKM.txt",sep="\t",index=False)
    display(df.query('Symbol=="Nlk"'))
    print(inp_path.split(".txt")[0]+"_RPKM.txt was generated")

samples = ["MM_forbrain_E13.5_H3K27ac.bam","MM_forbrain_E13.5_H3K4me1.bam","MM_forbrain_E13.5_input.bam"]
# samtools view -c *bam
mapped_reads = [56668760,111757494,120459235]

inp_path="./MM_forbrain_E13.5_epigenome_tx_counts_int.txt"
calc_rpkm(inp_path,samples,mapped_reads)

samples = ["MM_forbrain_E13.5_ATAC.bam"]
# samtools view -c *bam
mapped_reads = [147078772]

inp_path="./MM_forbrain_E13.5_epigenome_tx_counts_atac_int.txt"
calc_rpkm(inp_path,samples,mapped_reads)

,chr,start,end,ID,score,strand,start2,end2,exons,blocks,bl_size,bl_start,Geneid,Symbol
0,chr1,3143475,3144545,4933401J01Rik:ENSMUSG00000102693.2:ENSMUST00000193812.2,0,+,3143475,3144545,0,1,"1070,","0,",ENSMUSG00000102693.2,4933401J01Rik
1,chr1,3172238,3172348,Gm26206:ENSMUSG00000064842.3:ENSMUST00000082908.3,0,+,3172238,3172348,0,1,"110,","0,",ENSMUSG00000064842.3,Gm26206


,Symbol,Geneid,Chr,Start,End,Strand,Length,MM_forbrain_E13.5_H3K27ac.bam,MM_forbrain_E13.5_H3K4me1.bam,MM_forbrain_E13.5_input.bam,RPM_MM_forbrain_E13.5_H3K27ac,RPKM_MM_forbrain_E13.5_H3K27ac,RPM_MM_forbrain_E13.5_H3K4me1,RPKM_MM_forbrain_E13.5_H3K4me1,RPM_MM_forbrain_E13.5_input,RPKM_MM_forbrain_E13.5_input
3869,Nlk,ENSMUSG00000017376.16,chr11,78457994,78585699,-,127706,3549,8003,6666,62.627098,0.490401,71.61041,0.560744,55.338223,0.433325


../SFPQ/BELD/MM_forbrain_E13.5_epigenome_tx_counts_int_RPKM.txt was generated


,Symbol,Geneid,Chr,Start,End,Strand,Length,MM_forbrain_E13.5_ATAC.bam,RPM_MM_forbrain_E13.5_ATAC,RPKM_MM_forbrain_E13.5_ATAC
3869,Nlk,ENSMUSG00000017376.16,chr11,78457994,78585699,-,127706,3532,24.014342,0.188044


../SFPQ/BELD/MM_forbrain_E13.5_epigenome_tx_counts_atac_int_RPKM.txt was generated
